# 🗺️ Peta 3 Provinsi untuk Sidang
**Banten, DKI Jakarta, Jawa Barat — 41 Kabupaten/Kota**

Menggunakan shapefile GADM level 2 (`gadm41_IDN_2.shp`) yang sudah ada di folder `web/`.

Output: beberapa versi gambar siap pakai untuk presentasi sidang.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import numpy as np
import os

# ── Path shapefile ──────────────────────────────────────────────────────────
SHP_PATH = os.path.join(os.path.dirname(os.getcwd()), 'web', 'gadm41_IDN_2.shp')
OUT_DIR   = os.getcwd()   # simpan di folder debug/

print('Shapefile:', SHP_PATH)
print('Output ke :', OUT_DIR)

In [ ]:
# ── Load & Filter ───────────────────────────────────────────────────────────
gdf_raw = gpd.read_file(SHP_PATH)

PROV_NAMES = ['Banten', 'Jakarta Raya', 'Jawa Barat']
gdf = gdf_raw[gdf_raw['NAME_1'].isin(PROV_NAMES)].copy()

# Hapus Waduk Cirata (badan air, bukan kab/kota)
gdf = gdf[gdf['TYPE_2'] != 'Water Body'].copy()

PROV_DISPLAY = {
    'Banten':       'BANTEN',
    'Jakarta Raya': 'DKI JAKARTA',
    'Jawa Barat':   'JAWA BARAT',
}
gdf['PROV_DISPLAY'] = gdf['NAME_1'].map(PROV_DISPLAY)

# Background Pulau Jawa untuk inset (semua 6 provinsi)
JAWA_PROV = ['Banten','Jakarta Raya','Jawa Barat',
             'Jawa Tengah','Daerah Istimewa Yogyakarta','Jawa Timur']
gdf_jawa = gdf_raw[gdf_raw['NAME_1'].isin(JAWA_PROV)].copy()
gdf_jawa = gdf_jawa[gdf_jawa['TYPE_2'] != 'Water Body'].copy()

print(f'Fitur 3 provinsi: {len(gdf)}')
print(gdf.groupby('PROV_DISPLAY')['NAME_2'].count())
print(f'\nFitur background Jawa: {len(gdf_jawa)}')

In [ ]:
# ── Proyeksi ke UTM Zone 48S untuk centroid akurat ──────────────────────────
gdf_proj = gdf.to_crs('EPSG:32748')   # UTM 48S
gdf_proj['centroid'] = gdf_proj.geometry.centroid
gdf['cx'] = gdf_proj['centroid'].to_crs('EPSG:4326').x
gdf['cy'] = gdf_proj['centroid'].to_crs('EPSG:4326').y

# Batas peta
BOUNDS = gdf.total_bounds   # [minx, miny, maxx, maxy]
PAD_X, PAD_Y = 0.15, 0.12
XLIM = (BOUNDS[0] - PAD_X, BOUNDS[2] + PAD_X)
YLIM = (BOUNDS[1] - PAD_Y, BOUNDS[3] + PAD_Y)

print('Batas peta (lon/lat):', BOUNDS.round(3))

In [ ]:
# ── Palet warna ─────────────────────────────────────────────────────────────
PROV_COLORS = {
    'DKI JAKARTA': '#1E88E5',   # biru
    'JAWA BARAT' : '#2E7D32',   # hijau tua
    'BANTEN'     : '#E65100',   # oranye tua
}
PROV_LIGHT = {
    'DKI JAKARTA': '#BBDEFB',
    'JAWA BARAT' : '#C8E6C9',
    'BANTEN'     : '#FFE0B2',
}
gdf['fill_color'] = gdf['PROV_DISPLAY'].map(PROV_LIGHT)

---
## Versi 1 — Light (cocok untuk slide terang)

In [ ]:
def plot_peta_light(gdf, save_path=None, show_labels=True, label_size=5.5):
    fig, ax = plt.subplots(figsize=(12, 7.5), dpi=180)
    fig.patch.set_facecolor('#F8FAFF')
    ax.set_facecolor('#D6EAF8')   # laut biru muda

    # ── Gambar tiap provinsi ─────────────────────────────────────────────────
    for prov, color_light in PROV_LIGHT.items():
        sub = gdf[gdf['PROV_DISPLAY'] == prov]
        sub.plot(ax=ax,
                 color=color_light,
                 edgecolor='#ffffff',
                 linewidth=0.7,
                 zorder=2)

    # ── Batas provinsi tebal ─────────────────────────────────────────────────
    for prov, color in PROV_COLORS.items():
        sub = gdf[gdf['PROV_DISPLAY'] == prov]
        dissolved = sub.dissolve()
        dissolved.boundary.plot(ax=ax,
                                color=color,
                                linewidth=2.2,
                                zorder=4)

    # ── Label kabupaten/kota ─────────────────────────────────────────────────
    if show_labels:
        for _, row in gdf.iterrows():
            name = row['NAME_2']
            # Singkat beberapa nama panjang
            SHORT = {
                'Kota Jakarta Selatan': 'Jak. Selatan',
                'Kota Jakarta Timur'  : 'Jak. Timur',
                'Kota Jakarta Pusat'  : 'Jak. Pusat',
                'Kota Jakarta Barat'  : 'Jak. Barat',
                'Kota Jakarta Utara'  : 'Jak. Utara',
                'Kota Tangerang Selatan': 'Tangsel',
                'Kota Tasikmalaya'    : 'Kota Tasik',
                'Bandung Barat'       : 'Bdg. Barat',
            }
            label = SHORT.get(name, name.replace('Kota ', 'K.'))
            c = PROV_COLORS[row['PROV_DISPLAY']]
            ax.text(
                row['cx'], row['cy'], label,
                fontsize=label_size, ha='center', va='center',
                color=c, fontweight='bold',
                path_effects=[pe.withStroke(linewidth=1.8, foreground='white')],
                zorder=6
            )

    # ── Label nama provinsi ──────────────────────────────────────────────────
    PROV_LABEL_POS = {
        'DKI JAKARTA': (106.820, -6.17),
        'BANTEN'     : (106.07,  -6.70),
        'JAWA BARAT' : (107.60,  -6.72),
    }
    for prov, (lx, ly) in PROV_LABEL_POS.items():
        c = PROV_COLORS[prov]
        ax.text(lx, ly, prov,
                fontsize=10, fontweight='black', ha='center',
                color='white',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=c, alpha=0.88, edgecolor='none'),
                zorder=8)

    # ── Legenda ──────────────────────────────────────────────────────────────
    patches = [mpatches.Patch(facecolor=PROV_LIGHT[p], edgecolor=PROV_COLORS[p],
                               linewidth=1.5, label=p)
               for p in ['DKI JAKARTA','BANTEN','JAWA BARAT']]
    leg = ax.legend(handles=patches, loc='lower left',
                    fontsize=8.5, framealpha=0.9,
                    edgecolor='#cccccc', facecolor='white',
                    title='Provinsi', title_fontsize=9)
    leg.get_frame().set_linewidth(0.8)

    # ── Judul & anotasi ──────────────────────────────────────────────────────
    ax.set_title(
        'Wilayah Pengamatan: Banten, DKI Jakarta, dan Jawa Barat\n'
        '(41 Kabupaten/Kota, 2020–2024)',
        fontsize=12, fontweight='bold', pad=10,
        color='#1a2744'
    )
    ax.text(0.99, 0.01, 'Sumber: GADM v4.1 | BPS RI',
            transform=ax.transAxes, fontsize=6.5,
            ha='right', va='bottom', color='#888888')

    # ── Axes styling ─────────────────────────────────────────────────────────
    ax.set_xlim(XLIM); ax.set_ylim(YLIM)
    ax.set_xlabel('Bujur (°BT)', fontsize=8)
    ax.set_ylabel('Lintang (°LS)', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, linewidth=0.3, color='#bbd5e8', zorder=1)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2); spine.set_color('#1a2744')

    # ── Arah Utara ───────────────────────────────────────────────────────────
    ax.annotate('N', xy=(0.97, 0.18), xytext=(0.97, 0.12),
                xycoords='axes fraction',
                fontsize=11, fontweight='black', ha='center',
                color='#1a2744',
                arrowprops=dict(arrowstyle='->', color='#1a2744', lw=2),
                annotation_clip=False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=250, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        print(f'Tersimpan: {save_path}')
    plt.show()

plot_peta_light(gdf, save_path=os.path.join(OUT_DIR, 'peta_sidang_light.png'))

---
## Versi 2 — Dark (cocok untuk slide gelap / navy)

In [ ]:
def plot_peta_dark(gdf, save_path=None, show_labels=True, label_size=5.5):
    DARK_BG   = '#0f172a'   # dark navy
    OCEAN_COL = '#1e3a5f'   # navy lebih terang untuk laut
    BORDER_KAB = '#0f172a'

    PROV_DARK_FILL = {
        'DKI JAKARTA': '#1565C0',   # biru medium
        'JAWA BARAT' : '#1B5E20',   # hijau tua
        'BANTEN'     : '#BF360C',   # merah-oranye
    }
    PROV_DARK_EDGE = {
        'DKI JAKARTA': '#64B5F6',
        'JAWA BARAT' : '#81C784',
        'BANTEN'     : '#FFAB91',
    }

    fig, ax = plt.subplots(figsize=(12, 7.5), dpi=180)
    fig.patch.set_facecolor(DARK_BG)
    ax.set_facecolor(OCEAN_COL)

    for prov, fill in PROV_DARK_FILL.items():
        sub = gdf[gdf['PROV_DISPLAY'] == prov]
        sub.plot(ax=ax, color=fill, edgecolor='#1e293b', linewidth=0.5, zorder=2)

    # Batas provinsi
    for prov, edge in PROV_DARK_EDGE.items():
        sub = gdf[gdf['PROV_DISPLAY'] == prov]
        sub.dissolve().boundary.plot(ax=ax, color=edge, linewidth=2.0, zorder=4)

    if show_labels:
        SHORT = {
            'Kota Jakarta Selatan': 'Jak. Sel',
            'Kota Jakarta Timur'  : 'Jak. Tim',
            'Kota Jakarta Pusat'  : 'Jak. Pus',
            'Kota Jakarta Barat'  : 'Jak. Bar',
            'Kota Jakarta Utara'  : 'Jak. Ut',
            'Kota Tangerang Selatan': 'Tangsel',
            'Kota Tasikmalaya'    : 'K.Tasik',
            'Bandung Barat'       : 'Bdg.Bar',
        }
        for _, row in gdf.iterrows():
            name   = row['NAME_2']
            label  = SHORT.get(name, name.replace('Kota ', 'K.'))
            c_edge = PROV_DARK_EDGE[row['PROV_DISPLAY']]
            ax.text(
                row['cx'], row['cy'], label,
                fontsize=label_size, ha='center', va='center',
                color=c_edge, fontweight='bold',
                path_effects=[pe.withStroke(linewidth=1.8, foreground='#0f172a')],
                zorder=6
            )

    PROV_LABEL_POS = {
        'DKI JAKARTA': (106.820, -6.17),
        'BANTEN'     : (106.07,  -6.70),
        'JAWA BARAT' : (107.60,  -6.72),
    }
    for prov, (lx, ly) in PROV_LABEL_POS.items():
        c = PROV_DARK_EDGE[prov]
        ax.text(lx, ly, prov,
                fontsize=10, fontweight='black', ha='center',
                color='#0f172a',
                bbox=dict(boxstyle='round,pad=0.3',
                          facecolor=c, alpha=0.9, edgecolor='none'),
                zorder=8)

    patches = [
        mpatches.Patch(facecolor=PROV_DARK_FILL[p], edgecolor=PROV_DARK_EDGE[p],
                        linewidth=1.5, label=p)
        for p in ['DKI JAKARTA','BANTEN','JAWA BARAT']
    ]
    leg = ax.legend(handles=patches, loc='lower left',
                    fontsize=8.5, framealpha=0.92,
                    edgecolor='#334155', facecolor='#1e293b',
                    labelcolor='#e2e8f0',
                    title='Provinsi', title_fontsize=9)
    leg.get_title().set_color('#94a3b8')

    ax.set_title(
        'Wilayah Pengamatan: Banten, DKI Jakarta, dan Jawa Barat\n'
        '(41 Kabupaten/Kota, 2020–2024)',
        fontsize=12, fontweight='bold', pad=10, color='#e2e8f0'
    )
    ax.text(0.99, 0.01, 'Sumber: GADM v4.1 | BPS RI',
            transform=ax.transAxes, fontsize=6.5,
            ha='right', va='bottom', color='#64748b')
    ax.annotate('N', xy=(0.97, 0.18), xytext=(0.97, 0.12),
                xycoords='axes fraction',
                fontsize=11, fontweight='black', ha='center', color='#e2e8f0',
                arrowprops=dict(arrowstyle='->', color='#e2e8f0', lw=2),
                annotation_clip=False)

    ax.set_xlim(XLIM); ax.set_ylim(YLIM)
    ax.set_xlabel('Bujur (°BT)', fontsize=8, color='#94a3b8')
    ax.set_ylabel('Lintang (°LS)', fontsize=8, color='#94a3b8')
    ax.tick_params(labelsize=7, colors='#94a3b8')
    ax.grid(True, linewidth=0.25, color='#1e3a5f', zorder=1)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0); spine.set_color('#334155')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=250, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        print(f'Tersimpan: {save_path}')
    plt.show()

plot_peta_dark(gdf, save_path=os.path.join(OUT_DIR, 'peta_sidang_dark.png'))

---
## Versi 3 — Minimalis Tanpa Label (untuk slide yang padat teks)

In [ ]:
def plot_peta_minimal(gdf, save_path=None):
    """Peta bersih tanpa label kab/kota — hanya label provinsi."""
    fig, ax = plt.subplots(figsize=(8, 5.5), dpi=180)
    fig.patch.set_facecolor('#F0F4FF')
    ax.set_facecolor('#C5D8EE')

    FILL = {'DKI JAKARTA':'#90CAF9','JAWA BARAT':'#A5D6A7','BANTEN':'#FFCC80'}
    EDGE = {'DKI JAKARTA':'#1565C0','JAWA BARAT':'#2E7D32','BANTEN':'#E65100'}

    for prov in ['DKI JAKARTA','BANTEN','JAWA BARAT']:
        sub = gdf[gdf['PROV_DISPLAY'] == prov]
        sub.plot(ax=ax, color=FILL[prov],
                 edgecolor='white', linewidth=0.5, zorder=2)
        sub.dissolve().boundary.plot(ax=ax, color=EDGE[prov],
                                     linewidth=2.0, zorder=4)

    # Hanya label provinsi besar
    PROV_LABEL_POS = {
        'DKI JAKARTA': (106.820, -6.17, 9),
        'BANTEN'     : (106.07,  -6.70, 9),
        'JAWA BARAT' : (107.60,  -6.72, 9),
    }
    for prov, (lx, ly, fs) in PROV_LABEL_POS.items():
        ax.text(lx, ly, prov, fontsize=fs, fontweight='black',
                ha='center', color='white',
                bbox=dict(boxstyle='round,pad=0.35',
                          facecolor=EDGE[prov], alpha=0.9, edgecolor='none'),
                zorder=8)

    # Anotasi jumlah kab/kota per provinsi
    PROV_N = gdf.groupby('PROV_DISPLAY').size().to_dict()
    PROV_N_POS = {
        'DKI JAKARTA': (106.820, -6.27),
        'BANTEN'     : (106.07,  -6.83),
        'JAWA BARAT' : (107.60,  -6.85),
    }
    for prov, (lx, ly) in PROV_N_POS.items():
        n = PROV_N.get(prov, 0)
        ax.text(lx, ly, f'{n} kab/kota', fontsize=7.5,
                ha='center', color=EDGE[prov], fontstyle='italic',
                path_effects=[pe.withStroke(linewidth=2, foreground='white')],
                zorder=7)

    ax.set_title('Tiga Provinsi Jawa Bagian Barat  (N = 41)',
                 fontsize=11, fontweight='bold', pad=8, color='#1a2744')
    ax.text(0.99, 0.01, 'Sumber: GADM v4.1',
            transform=ax.transAxes, fontsize=6, ha='right', va='bottom', color='#888')
    ax.annotate('N', xy=(0.97, 0.2), xytext=(0.97, 0.13),
                xycoords='axes fraction', fontsize=10, fontweight='black',
                ha='center', color='#1a2744',
                arrowprops=dict(arrowstyle='->', color='#1a2744', lw=1.8),
                annotation_clip=False)

    patches = [mpatches.Patch(facecolor=FILL[p], edgecolor=EDGE[p],
                               linewidth=1.5, label=p)
               for p in ['DKI JAKARTA','BANTEN','JAWA BARAT']]
    ax.legend(handles=patches, loc='lower left', fontsize=7.5,
              framealpha=0.9, edgecolor='#ccc', facecolor='white')

    ax.set_xlim(XLIM); ax.set_ylim(YLIM)
    ax.tick_params(labelsize=6.5)
    ax.grid(True, linewidth=0.25, color='#b0c8e0', zorder=1)
    ax.set_xlabel('Bujur (°BT)', fontsize=7)
    ax.set_ylabel('Lintang (°LS)', fontsize=7)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=250, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        print(f'Tersimpan: {save_path}')
    plt.show()

plot_peta_minimal(gdf, save_path=os.path.join(OUT_DIR, 'peta_sidang_minimal.png'))

---
## Versi 4 — Inset: Peta Lokasi dalam Pulau Jawa

In [ ]:
def plot_peta_dengan_inset(gdf, gdf_jawa, save_path=None, dark=False):
    """Peta utama (kiri) + inset Pulau Jawa (kanan atas) + info box (kanan bawah)."""

    BG      = '#0f172a' if dark else '#F0F4FF'
    OCEAN   = '#1e3a5f' if dark else '#C5D8EE'
    TXT_COL = '#e2e8f0' if dark else '#1a2744'
    FILL = ({'DKI JAKARTA':'#1565C0','JAWA BARAT':'#1B5E20','BANTEN':'#BF360C'}
            if dark else
            {'DKI JAKARTA':'#90CAF9','JAWA BARAT':'#A5D6A7','BANTEN':'#FFCC80'})
    EDGE = {'DKI JAKARTA': '#64B5F6' if dark else '#1565C0',
            'JAWA BARAT' : '#81C784' if dark else '#2E7D32',
            'BANTEN'     : '#FFAB91' if dark else '#E65100'}

    SHORT = {
        'Kota Jakarta Selatan':'Jak. Sel','Kota Jakarta Timur':'Jak. Tim',
        'Kota Jakarta Pusat':'Jak. Pus','Kota Jakarta Barat':'Jak. Bar',
        'Kota Jakarta Utara':'Jak. Ut','Kota Tangerang Selatan':'Tangsel',
        'Kota Tasikmalaya':'K.Tasik','Bandung Barat':'Bdg.Bar',
    }

    fig = plt.figure(figsize=(13, 8), dpi=180)
    fig.patch.set_facecolor(BG)

    # ── Axes utama (70% lebar kiri) ──────────────────────────────────────────
    ax_main = fig.add_axes([0.02, 0.05, 0.69, 0.90])
    ax_main.set_facecolor(OCEAN)

    for prov in ['DKI JAKARTA','BANTEN','JAWA BARAT']:
        sub = gdf[gdf['PROV_DISPLAY'] == prov]
        sub.plot(ax=ax_main, color=FILL[prov],
                 edgecolor='#0f172a' if dark else 'white',
                 linewidth=0.5, zorder=2)
        sub.dissolve().boundary.plot(ax=ax_main, color=EDGE[prov],
                                     linewidth=2.0, zorder=4)

    for _, row in gdf.iterrows():
        label = SHORT.get(row['NAME_2'], row['NAME_2'].replace('Kota ', 'K.'))
        ax_main.text(row['cx'], row['cy'], label,
                     fontsize=4.8, ha='center', va='center',
                     color=EDGE[row['PROV_DISPLAY']], fontweight='bold',
                     path_effects=[pe.withStroke(
                         linewidth=1.5, foreground='#0f172a' if dark else 'white')],
                     zorder=6)

    for prov, (lx, ly) in [('DKI JAKARTA',(106.82,-6.17)),
                            ('BANTEN',(106.07,-6.70)),
                            ('JAWA BARAT',(107.60,-6.72))]:
        ax_main.text(lx, ly, prov, fontsize=9.5, fontweight='black', ha='center',
                     color='#0f172a' if dark else 'white',
                     bbox=dict(boxstyle='round,pad=0.3',
                               facecolor=EDGE[prov], alpha=0.9, edgecolor='none'),
                     zorder=8)

    ax_main.set_xlim(XLIM); ax_main.set_ylim(YLIM)
    ax_main.set_title(
        'Wilayah Pengamatan: Banten, DKI Jakarta, dan Jawa Barat\n'
        '(41 Kabupaten/Kota, 2020–2024)',
        fontsize=11, fontweight='bold', pad=8, color=TXT_COL)
    ax_main.set_xlabel('Bujur (°BT)', fontsize=7.5,
                        color='#94a3b8' if dark else None)
    ax_main.set_ylabel('Lintang (°LS)', fontsize=7.5,
                        color='#94a3b8' if dark else None)
    ax_main.tick_params(labelsize=6.5, colors='#94a3b8' if dark else 'black')
    ax_main.grid(True, linewidth=0.25,
                  color='#1e3a5f' if dark else '#b0c8e0', zorder=1)
    for sp in ax_main.spines.values():
        sp.set_color('#334155' if dark else '#1a2744'); sp.set_linewidth(1.2)

    patches = [mpatches.Patch(facecolor=FILL[p], edgecolor=EDGE[p],
                               linewidth=1.5, label=p)
               for p in ['DKI JAKARTA','BANTEN','JAWA BARAT']]
    leg = ax_main.legend(handles=patches, loc='lower left', fontsize=7.5,
                         framealpha=0.9,
                         edgecolor='#334155' if dark else '#ccc',
                         facecolor='#1e293b' if dark else 'white',
                         labelcolor='#e2e8f0' if dark else 'black')
    if dark: leg.get_frame().set_linewidth(0.8)

    ax_main.annotate('N', xy=(0.97, 0.2), xytext=(0.97, 0.13),
                     xycoords='axes fraction', fontsize=10, fontweight='black',
                     ha='center', color=TXT_COL,
                     arrowprops=dict(arrowstyle='->', color=TXT_COL, lw=1.8),
                     annotation_clip=False)
    ax_main.text(0.98, 0.01, 'Sumber: GADM v4.1 | BPS RI',
                 transform=ax_main.transAxes, fontsize=5.5,
                 ha='right', va='bottom', color='#64748b' if dark else '#888')

    # ── Inset: konteks Pulau Jawa (kanan atas) ───────────────────────────────
    ax_ins = fig.add_axes([0.72, 0.55, 0.26, 0.40])
    ax_ins.set_facecolor('#1e3a5f' if dark else '#C5D8EE')

    # Background semua kabupaten Jawa
    gdf_jawa.plot(ax=ax_ins,
                  color='#2d3748' if dark else '#dde8f0',
                  edgecolor='#475569' if dark else '#9ab0c8',
                  linewidth=0.3, zorder=1)

    # Highlight 3 provinsi yang diteliti
    for prov in ['DKI JAKARTA','BANTEN','JAWA BARAT']:
        sub = gdf[gdf['PROV_DISPLAY'] == prov]
        sub.dissolve().plot(ax=ax_ins,
                            color=EDGE[prov], alpha=0.80,
                            edgecolor='white', linewidth=0.8, zorder=3)

    # Kotak batas peta utama sebagai referensi
    from matplotlib.patches import Rectangle
    rect = Rectangle((XLIM[0], YLIM[0]),
                      XLIM[1]-XLIM[0], YLIM[1]-YLIM[0],
                      linewidth=1.2, edgecolor='yellow',
                      facecolor='none', linestyle='--', zorder=5)
    ax_ins.add_patch(rect)

    ax_ins.set_xlim(104.7, 115.8)
    ax_ins.set_ylim(-8.9, -5.6)
    ax_ins.set_title('Lokasi di Pulau Jawa', fontsize=6.5,
                     color=TXT_COL, pad=3, fontweight='bold')
    ax_ins.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for sp in ax_ins.spines.values():
        sp.set_color('#334155' if dark else '#8898aa'); sp.set_linewidth(0.8)

    # ── Info box (kanan bawah) ────────────────────────────────────────────────
    PROV_N = gdf.groupby('PROV_DISPLAY').size().to_dict()
    info = (
        'Data Penelitian\n'
        '─────────────────────\n'
        f'• BANTEN        : {PROV_N.get("BANTEN",0)} kab/kota\n'
        f'• DKI JAKARTA   : {PROV_N.get("DKI JAKARTA",0)} kab/kota\n'
        f'• JAWA BARAT    : {PROV_N.get("JAWA BARAT",0)} kab/kota\n'
        '─────────────────────\n'
        '  Total N = 41 wilayah\n'
        '  Periode: 2020 – 2024\n'
        '─────────────────────\n'
        '  Sumber: BPS RI\n'
        '  Peta: GADM v4.1'
    )
    ax_box = fig.add_axes([0.72, 0.05, 0.26, 0.46])
    ax_box.axis('off')
    ax_box.text(0.05, 0.97, info, transform=ax_box.transAxes,
                fontsize=8.5, va='top', color=TXT_COL,
                fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.7',
                          facecolor='#1e293b' if dark else '#f0f4ff',
                          edgecolor='#334155' if dark else '#c0d0e0',
                          linewidth=0.9))

    if save_path:
        plt.savefig(save_path, dpi=250, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        print(f'Tersimpan: {save_path}')
    plt.show()

# Render kedua versi
plot_peta_dengan_inset(gdf, gdf_jawa, dark=False,
    save_path=os.path.join(OUT_DIR, 'peta_sidang_inset_light.png'))

plot_peta_dengan_inset(gdf, gdf_jawa, dark=True,
    save_path=os.path.join(OUT_DIR, 'peta_sidang_inset_dark.png'))

---
## Ekspor GeoJSON (opsional — untuk kebutuhan web/leaflet)

In [ ]:
# Ekspor GeoJSON (bisa digunakan di Leaflet, D3, atau landing page)
import json

gdf_export = gdf[['NAME_1','NAME_2','PROV_DISPLAY','geometry']].copy()
gdf_export = gdf_export.rename(columns={
    'NAME_1':'provinsi_gadm',
    'NAME_2':'kab_kota',
    'PROV_DISPLAY':'provinsi'
})
# Sederhanakan geometri sedikit
gdf_export['geometry'] = gdf_export['geometry'].simplify(0.003, preserve_topology=True)

geojson_path = os.path.join(OUT_DIR, '3provinsi_41kota.geojson')
gdf_export.to_file(geojson_path, driver='GeoJSON')
print(f'GeoJSON tersimpan: {geojson_path}')
print(f'Jumlah fitur: {len(gdf_export)}')
print('Kolom:', list(gdf_export.columns))

---
## Ringkasan Output

| File | Keterangan |
|------|------------|
| `peta_sidang_light.png` | Peta terang dengan label semua kab/kota |
| `peta_sidang_dark.png` | Peta gelap (cocok slide navy/dark) |
| `peta_sidang_minimal.png` | Versi ringkas, hanya label provinsi |
| `peta_sidang_inset_light.png` | Peta terang + inset Pulau Jawa + info box |
| `peta_sidang_inset_dark.png` | Peta gelap + inset Pulau Jawa + info box |
| `3provinsi_41kota.geojson` | GeoJSON untuk kebutuhan web |

> **Saran untuk sidang:** Gunakan `peta_sidang_inset_light.png` atau `peta_sidang_minimal.png`  
> tergantung tema slide. Inset versi sudah cocok dengan placeholder di slide Latar Belakang.
> Ganti dengan `_dark` jika tema slide gelap/navy.